**Import Required Libraries:**

In [36]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, Dropout
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.utils import to_categorical

**Input Text:**

In [13]:
sample_text = """
The sun rose slowly over the mountains, casting long golden rays across the valley below.
Birds began to sing their morning songs as the world slowly woke from its slumber.
A gentle breeze moved through the tall grass, bending each blade in a silent dance.
The river sparkled in the early light, its waters rushing over smooth stones with a soft murmur.
Far in the distance, a lone wolf howled, echoing through the vast wilderness.
Children laughed as they ran through the meadow, chasing butterflies and rolling in the soft green grass.
An old farmer walked along the dirt path, carrying a basket filled with fresh vegetables from his garden.
The oak tree stood tall at the edge of the field, its roots deep and its branches wide.
A robin perched on the fence post and tilted its head, watching the world with curious eyes.
Clouds drifted lazily across the blue sky, their shapes shifting like the thoughts of a dreaming mind.
The village baker began his work before dawn, filling the air with the warm scent of fresh bread.
Market day brought people from all the nearby towns, their carts piled high with goods and laughter.
A young girl sat by the stream, dipping her toes into the cold water, listening to the stones.
Her grandmother had told her stories of this place, of the magic that lived in the old forest nearby.
The storyteller sat by the fire, and the children gathered close, their eyes wide with wonder.
Every story began the same way, with a traveler who walked into an unknown land seeking something lost.
The hero was never the strongest, but always the most curious and the most kind.
Mountains were crossed, rivers were forged, and dark forests were navigated with wit and heart.
At the end of every journey, the traveler found that what was sought had been within all along.
The fire crackled and popped as the last ember glowed orange in the dark night.
Stars filled the sky like scattered salt on a dark cloth, each one a story of its own.
An astronomer peered through his telescope, tracing the ancient paths of planets and comets.
He wrote in his notebook the coordinates of a newly discovered star, his hand trembling with excitement.
Science and wonder are not so different, he believed, both driven by questions with no simple answer.
The library was the oldest building in the city, its stone walls worn smooth by centuries of hands.
Books lined every shelf from floor to ceiling, their spines a mosaic of colors and languages.
A scholar sat at the reading table, surrounded by open volumes, lost in a world of ideas.
Knowledge, she believed, was the only true inheritance one generation could give the next.
She turned the page carefully, for the paper was thin and the words upon it were irreplaceable.
Rain began to fall outside, tapping softly against the tall windows of the old reading room.
The smell of old paper and rain mixed together in a way that felt timeless and peaceful.
Thunder rolled in the distance, and lightning briefly illuminated the dark clouds on the horizon.
A sailor on the dock looked up at the sky and tightened the ropes on his small wooden boat.
The sea could change in an instant, he knew from long experience and too many close calls.
He had sailed to places where the stars were different and the wind spoke in unfamiliar tongues.
Each voyage taught him something new about the world and something deeper about himself.
The port town was full of people with stories, every face a map of somewhere else.
A woman in a red coat stood at the water's edge, watching the horizon as if expecting someone.
Her letters had gone unanswered for months, but still she came every evening to watch for ships.
Hope is a quiet and persistent thing, refusing to be extinguished by silence or distance.
The clock tower in the center of town struck seven, and the streets began to fill with evening walkers.
Couples strolled arm in arm, and children on bicycles rang their bells as they sped past the cafes.
The smell of coffee and garlic drifted from open doorways into the cool evening air.
A musician sat on the corner, his fingers dancing over the strings of an old guitar.
People paused to listen, dropping coins into his case, some staying longer than they had planned.
Music has the power to stop time, to pull a person out of their worries and into the present.
The melody was simple but deep, like a river that looks calm on the surface but runs strong below.
Applause rippled through the small crowd, and the musician smiled without breaking his rhythm.
In the park, an artist set up her easel and began to paint the evening light on the pond.
She mixed colors intuitively, trusting her hands more than her eyes to find the right tone.
Art is not copying what you see but expressing what you feel when you look at the world.
A child stopped beside her and asked what she was painting, and she said, the light on the water.
The child looked at the canvas and then at the pond and nodded seriously, as if this made perfect sense.
Imagination is something children carry naturally, before the world teaches them to doubt themselves.
The evening deepened into night, and the city lights came on one by one like slow fireworks.
From the hilltop, the city below looked like a mirror of the sky above, full of quiet sparkling light.
A philosopher walked alone on the hilltop path, turning over ideas like stones in his hands.
What is real, he wondered, and what is only the story we tell ourselves to make sense of things.
These questions had no final answers, but the asking of them kept the mind sharp and the spirit humble.
He sat on a bench and watched the stars emerge, thinking that the universe was asking questions too.
The night deepened and the air grew cool, and somewhere below a dog barked once and was quiet.
A cat crept silently across a rooftop, pausing to observe the street below with complete indifference.
In a small apartment, a writer sat at her desk, staring at a blank page and waiting for words.
She had learned that the best thing to do when the words would not come was to keep sitting anyway.
Eventually, a sentence arrived, then another, and soon the page was filled with something real.
Writing is the art of turning silence into meaning, of shaping the chaos of thought into form.
She typed until midnight, then read what she had written and found it was better than she expected.
Tomorrow she would revise it, but tonight she allowed herself a moment of quiet satisfaction.
She made a cup of tea and stood at the window, watching the empty street below in the amber lamplight.
The city was quieter now, its daily noise replaced by the occasional passing car and distant sound.
Sleep came slowly for her, as it always did when the writing had gone well and her mind was still busy.
Dreams that night were vivid and strange, full of the characters she had been writing about all evening.
In the morning, she would return to the desk and the story would continue, one word at a time.
Every great journey, she believed, began with a single step, and every great story with a single word.
The sun would rise again over the mountains, and the valley would fill once more with golden light.
The birds would sing, the river would rush, and somewhere a child would laugh in the tall green grass.
And the world, ancient and endlessly new, would keep turning, keep wondering, keep beginning again.
"""

**Character Vocabulary:**

In [14]:
text = sample_text.lower().strip()
print(f"Total characters in text: {len(text)}")

Total characters in text: 7365


In [15]:
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
vocab_size = len(chars)
print(f"Vocabulary size (unique chars): {vocab_size}")

Vocabulary size (unique chars): 31


**Training Sequences:**

In [37]:
SEQ_LENGTH = 30
STEP = 2

In [17]:
X_data, y_data = [], []
for i in range(0, len(text) - SEQ_LENGTH, STEP):
    seq_in  = text[i : i + SEQ_LENGTH]
    seq_out = text[i + SEQ_LENGTH]
    X_data.append([char_to_idx[c] for c in seq_in])
    y_data.append(char_to_idx[seq_out])

In [18]:
n_samples = len(X_data)
print(f"Total training sequences: {n_samples}")

Total training sequences: 2442


**Reshape and one-hot encoding:**

In [19]:
X = np.array(X_data)
y = to_categorical(y_data, num_classes=vocab_size)

**Normalization:**

In [20]:
X_norm = X / float(vocab_size)
X_3d   = X_norm.reshape(n_samples, SEQ_LENGTH, 1)
print(f"X shape: {X_3d.shape}  |  y shape: {y.shape}")

X shape: (2442, 40, 1)  |  y shape: (2442, 31)


**Build RNN Model:**

In [38]:
def build_rnn_model(seq_len, vocab_size):
    model = Sequential([
        SimpleRNN(256, input_shape=(seq_len, 1), return_sequences=True),   # stacked
        Dropout(0.3),
        SimpleRNN(128, return_sequences=False),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dense(vocab_size, activation='softmax')
    ])
    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  metrics=['accuracy'])
    return model

In [39]:
rnn_model  = build_rnn_model(SEQ_LENGTH, vocab_size)

**Build LSTM Model:**

In [40]:
def build_lstm_model(seq_len, vocab_size):
    model = Sequential([
        LSTM(256, input_shape=(seq_len, 1), return_sequences=True),        # stacked
        Dropout(0.3),
        LSTM(128, return_sequences=False),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dense(vocab_size, activation='softmax')
    ])
    model.compile(loss='categorical_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  metrics=['accuracy'])
    return model

In [41]:
lstm_model = build_lstm_model(SEQ_LENGTH, vocab_size)

In [42]:
print("\n── RNN Model Summary ──")
rnn_model.summary()


── RNN Model Summary ──


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 30, 256)        │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 128)            │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 31)             │         3,999 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 135,839 (530.62 KB)

 Trainable params: 135,839 (530.62 KB)

 Non-trainable params: 0 (0.00 B)

In [43]:
print("\n── LSTM Model Summary ──")
lstm_model.summary()


── LSTM Model Summary ──


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 30, 256)        │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 30, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 128)            │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 31)             │         3,999 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 481,823 (1.84 MB)

 Trainable params: 481,823 (1.84 MB)

 Non-trainable params: 0 (0.00 B)

**Training the Models:**

In [45]:
callbacks = [
    ReduceLROnPlateau(monitor='loss', factor=0.5, patience=3,
                      min_lr=1e-5, verbose=1),
    EarlyStopping(monitor='loss', patience=7,
                  restore_best_weights=True, verbose=1)
]

In [44]:
EPOCHS     = 50
BATCH_SIZE = 64

In [46]:
print("\n🔵 Training Improved RNN model...")
rnn_history = rnn_model.fit(
    X_3d, y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)


🔵 Training Improved RNN model...
Epoch 1/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 8s 114ms/step - accuracy: 0.1324 - loss: 3.1486 - learning_rate: 0.0010
Epoch 2/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 4s 92ms/step - accuracy: 0.1462 - loss: 2.9872 - learning_rate: 0.0010
Epoch 3/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 4s 95ms/step - accuracy: 0.1598 - loss: 2.9431 - learning_rate: 0.0010
Epoch 4/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 5s 131ms/step - accuracy: 0.1568 - loss: 2.9327 - learning_rate: 0.0010
Epoch 5/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 4s 91ms/step - accuracy: 0.1584 - loss: 2.9514 - learning_rate: 0.0010
Epoch 6/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 6s 111ms/step - accuracy: 0.1771 - loss: 2.9264 - learning_rate: 0.0010
Epoch 7/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 4s 107ms/step - accuracy: 0.1710 - loss: 2.9256 - learning_rate: 0.0010
Epoch 8/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 5s 134ms/step - accuracy: 0.1420 - loss: 2.9516 - learning_rate: 0.0010
Epoch 9/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 5s 123ms/step - accuracy: 0.1624 - loss: 2.9376 - l

In [47]:
print("\n🟢 Training Improved LSTM model...")
lstm_history = lstm_model.fit(
    X_3d, y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)


🟢 Training Improved LSTM model...
Epoch 1/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 16s 309ms/step - accuracy: 0.1335 - loss: 3.1748 - learning_rate: 0.0010
Epoch 2/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 12s 311ms/step - accuracy: 0.1701 - loss: 2.9357 - learning_rate: 0.0010
Epoch 3/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 12s 317ms/step - accuracy: 0.1661 - loss: 2.9319 - learning_rate: 0.0010
Epoch 4/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 12s 310ms/step - accuracy: 0.1743 - loss: 2.9162 - learning_rate: 0.0010
Epoch 5/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 12s 311ms/step - accuracy: 0.1673 - loss: 2.9280 - learning_rate: 0.0010
Epoch 6/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 12s 311ms/step - accuracy: 0.1767 - loss: 2.9168 - learning_rate: 0.0010
Epoch 7/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 12s 310ms/step - accuracy: 0.1836 - loss: 2.8846 - learning_rate: 0.0010
Epoch 8/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 11s 292ms/step - accuracy: 0.1719 - loss: 2.9031 - learning_rate: 0.0010
Epoch 9/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 21s 311ms/step - accuracy: 0.1746 - los

**Text Generation Function:**

In [48]:
def generate_text(model, seed_text, gen_length=200, temperature=1.0):
    """
    Generate text from a trained model.
    temperature: lower = more predictable, higher = more creative
    """
    seed = seed_text.lower()
    if len(seed) < SEQ_LENGTH:
        seed = seed.rjust(SEQ_LENGTH)
    else:
        seed = seed[-SEQ_LENGTH:]

    generated = seed
    for _ in range(gen_length):
        x = np.array([char_to_idx.get(c, 0) for c in seed])
        x = x / float(vocab_size)
        x = x.reshape(1, SEQ_LENGTH, 1)

        preds = model.predict(x, verbose=0)[0]

        preds = np.log(preds + 1e-10) / temperature
        preds = np.exp(preds) / np.sum(np.exp(preds))

        next_idx  = np.random.choice(len(preds), p=preds)
        next_char = idx_to_char[next_idx]

        generated += next_char
        seed = seed[1:] + next_char

    return generated

**Generating Text & Validating:**

In [49]:
seed = "the sun rose slowly over the mountains cas"

print("\n" + "="*60)
print("📝 GENERATED TEXT — RNN Model")
print("="*60)
rnn_output = generate_text(rnn_model, seed, gen_length=300, temperature=0.8)
print(rnn_output)


📝 GENERATED TEXT — RNN Model
 slowly over the mountains casaaep a fe o ihoeh whi  is uivhe ateesefeeh  eegdeiheeo eiteh l ssdeahe,ae fii aet e ies tne ns i led  rno  te einrhseetsreeofearedio.e oiblsu eepe tnaettle kern n h ,ndyda  iber s  ashoro epe    isea tueahts.ptdiuti  w  sadho rtshalpsfebe ed hdetfe astiweru  sowhornoshhs   g, aat cli ese ro  ibaateo


In [50]:
seed = "the sun rose slowly over the mountains cas"

print("\n" + "="*60)
print("📝 GENERATED TEXT — LSTM Model")
print("="*60)
lstm_output = generate_text(lstm_model, seed, gen_length=300, temperature=0.8)
print(lstm_output)


📝 GENERATED TEXT — LSTM Model
 slowly over the mountains casbt  s t oro oi lctrse a suhegl td fhhsntesrde lnnohs histoa rfarldlfeh eisreatodshasaapaehst  ha a v
it eefoehto nvotrduwhsnwwes atwn tliussc  nee weosn gita  w to ehroedriieh.nveo nik as  nnbhautsttpwewcsned oaeeee  liiee eets
eanaeirh arfotu rta. sr  tnole w  isete. settdfi hnncrea dnoh  syota as,


**Validation Summary:**

In [51]:
rnn_final_acc  = rnn_history.history['accuracy'][-1]
lstm_final_acc = lstm_history.history['accuracy'][-1]
rnn_final_loss  = rnn_history.history['loss'][-1]
lstm_final_loss = lstm_history.history['loss'][-1]

In [52]:
print("\n" + "="*60)
print("✅ TRAINING SUMMARY")
print("="*60)
print(f"  RNN  — Final Loss: {rnn_final_loss:.4f}  |  Final Accuracy: {rnn_final_acc:.4f}")
print(f"  LSTM — Final Loss: {lstm_final_loss:.4f}  |  Final Accuracy: {lstm_final_acc:.4f}")
print("="*60)


✅ TRAINING SUMMARY
  RNN  — Final Loss: 2.9056  |  Final Accuracy: 0.1712
  LSTM — Final Loss: 2.8963  |  Final Accuracy: 0.1716
